![](https://img.shields.io/badge/PO.DAAC-Contribution-%20?color=grey&labelColor=blue)

> From the PO.DAAC Cookbook, to access the GitHub version of the notebook, follow [this link](https://github.com/podaac/tutorials/blob/master/notebooks/datasets/SWOT_Raster_Notebook_cloud.ipynb).


# Working with SWOT Level 2 Water Mask Raster Image Data Product:
## In AWS Cloud Version

#### *Authors: Nicholas Tarpinian, PO.DAAC | Catalina Taglialatela (JPL, PO.DAAC)*

*Last Updated: November 2026*

## Summary & Learning Objectives
### Notebook showcasing how to work with multiple files from the SWOT Raster Image data product version D in the cloud
- Utilizing the `earthaccess` Python package. For more information visit: https://nsidc.github.io/earthaccess/
- Option to query the new dataset based on user's choice; choosing between two resolutions either by  '100m' or '250m'.
- Visualizing multiple raster images on a single map.
- Stacking multiple raster images and creating a time dimension to analyze over time.
- Adjusting images based on quality flag

## Requirements

### 1. Compute environment 

This tutorial is written to run in the following environment:
- **AWS instance running in us-west-2**: NASA Earthdata Cloud data in S3 can be directly accessed via an s3fs session; this access is limited to requests made within the US West (Oregon) (code: `us-west-2`) AWS region.
- This workflow as written works on a 14.8 GB RAM, upto 4 CPU cloud compute instance type. Smaller instances tend to crash. 

### 2. Earthdata Login

An Earthdata Login account is required to access data, as well as discover restricted data, from the NASA Earthdata system. Thus, to access NASA data, you need Earthdata Login. Please visit https://urs.earthdata.nasa.gov to register and manage your Earthdata Login account. This account is free to create and only takes a moment to set up. 


### Import libraries

In [1]:
import io
import s3fs
import xarray as xr
import numpy as np
from datetime import datetime
from pathlib import Path
from shapely.geometry import Point, Polygon, box
import hvplot
import hvplot.xarray 
import earthaccess
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

## Authentication with earthaccess

In this notebook, we will be calling the authentication in the below cell.

In [2]:
auth = earthaccess.login()

### Search for SWOT Raster products using `earthaccess` 
Each dataset has its own unique collection concept ID. For the SWOT_L2_HR_Raster_D dataset, we can find the collection ID [here](https://www.earthdata.nasa.gov/data/catalog/pocloud-swot-l2-hr-raster-d-d).

For this tutorial, we are looking at the Lake Mead Reservoir in the United States.

We used [bbox finder](http://bboxfinder.com/) to get the exact coordinates for our area of interest.

In [3]:
raster_results = earthaccess.search_data(
    short_name = 'SWOT_L2_HR_Raster_D',
    bounding_box=(-115.112686,35.740939,-114.224167,36.937819),
    temporal =('2024-02-05 12:00:00', '2024-02-05 23:59:59'),
    granule_name = '*_100m_*',   #specify we are interested in the 100m standard raster
    count =200
)
print(f"Found {len(raster_results)} granules")

Found 1 granules


### Visualizing Multiple Tiles

Let's now visualize multiple raster tiles that we searched and explore the data.

Utilizing [xarray.open_mfdataset](https://docs.xarray.dev/en/stable/generated/xarray.open_mfdataset.html) which supports the opening of multiple files.

In [4]:
ds = xr.open_mfdataset(earthaccess.open(raster_results), engine='h5netcdf',combine='nested', concat_dim='x')
ds

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

<xarray.Dataset> Size: 606MB
Dimensions:                  (x: 1758, y: 1758)
Coordinates:
  * x                        (x) float64 14kB 4.635e+05 4.636e+05 ... 6.392e+05
  * y                        (y) float64 14kB 7.373e+06 7.373e+06 ... 7.548e+06
Data variables: (12/39)
    crs                      (x) object 14kB b'1' b'1' b'1' ... b'1' b'1' b'1'
    longitude                (y, x) float64 25MB dask.array<chunksize=(586, 586), meta=np.ndarray>
    latitude                 (y, x) float64 25MB dask.array<chunksize=(586, 586), meta=np.ndarray>
    wse                      (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    wse_qual                 (y, x) float32 12MB dask.array<chunksize=(1758, 1758), meta=np.ndarray>
    wse_qual_bitwise         (y, x) float64 25MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    ...                       ...
    load_tide_fes            (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    load_tide_got            (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    pole_tide                (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    model_dry_tropo_cor      (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    model_wet_tropo_cor      (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
    iono_cor_gim_ka          (y, x) float32 12MB dask.array<chunksize=(879, 879), meta=np.ndarray>
Attributes: (12/49)
    Conventions:                   CF-1.7
    title:                         Level 2 KaRIn High Rate Raster Data Product
    institution:                   JPL
    source:                        Ka-band radar interferometer
    history:                       2025-11-26T05:03:23Z : Creation
    platform:                      SWOT
    ...                            ...
    utm_zone_num:                  60
    mgrs_latitude_band:            W
    x_min:                         463500.0
    x_max:                         639200.0
    y_min:                         7372800.0
    y_max:                         7548500.0

In [5]:
raster_plot = ds.wse.hvplot.quadmesh(x='x', y='y', rasterize=True, title=f'SWOT Raster 100m: Lake Mead Reservoir')
raster_plot.opts(width=700, height=600, colorbar=True)

:DynamicMap   []
   :Image   [x,y]   (water surface elevation above geoid)

### Creating a Time Series 

SWOT Raster product does not include a time dimension, each file is a snapshot in time, but it can be inserted by extracting from the file name. 

1. Expand the time range of your `earthaccess` search to get an adequate range.
2. Filter the search results to granules whose footprint covers the area of interest, and keep the valid granules.
3. Extract the `datetime` from the s3 file name then concatenate based on the new time dimension.

In [6]:
# Area of interest: (lon_min, lat_min, lon_max, lat_max). Change this to your area of interest.
bbox = (-114.502048,36.060175,-114.390983,36.210182)
lon_min, lat_min, lon_max, lat_max = bbox

# Center of the area, used by the footprint filter
lon_c = (lon_min + lon_max) / 2
lat_c = (lat_min + lat_max) / 2

In [7]:
time_results = earthaccess.search_data(
    short_name = 'SWOT_L2_HR_Raster_D',
    bounding_box=bbox,
    temporal =('2024-01-25 00:00:00', '2024-03-04 23:59:59'),
    granule_name = '*_100m_*',
    count =200
)
print(f"Found {len(time_results)} granules")

Found 24 granules


In [8]:
fs_s3 = earthaccess.get_s3_filesystem(results=time_results)

In [9]:
# Get links list
raster = []
for g in time_results:
    for l in earthaccess.results.DataGranule.data_links(g, access='direct'):
        raster.append(l)

print(len(raster))
raster

24


['s3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM60W_N_x_x_x_010_073_140F_20240127T145008_20240127T145025_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM01W_N_x_x_x_010_138_018F_20240129T215328_20240129T215349_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM60W_N_x_x_x_010_194_019F_20240131T215450_20240131T215511_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM11S_N_x_x_x_010_205_109F_20240201T075048_20240201T075109_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM60V_N_x_x_x_010_222_021F_20240201T215601_20240201T215622_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster_D/SWOT_L2_HR_Raster_100m_UTM01W_N_x_x_x_010_267_137F_20240203T131000_20240203T131013_PGD0_01.nc',
 's3://podaac-swot-ops-cumulus-protected/SWOT_L2_HR_Raster

In [10]:
# Footprint filter: keep granules whose footprint polygon contains the point
aoi = box(lon_min, lat_min, lon_max, lat_max)

def footprint_covers(granule, aoi):
    """True if one of the granule's footprint polygons intersects the area of interest."""
    geometry = (granule.get("umm", {}).get("SpatialExtent", {})
                       .get("HorizontalSpatialDomain", {}).get("Geometry", {}))
    for polygon in geometry.get("GPolygons", []):
        points = [(p["Longitude"], p["Latitude"]) for p in polygon.get("Boundary", {}).get("Points", [])]
        if points and Polygon(points).intersects(aoi):
            return True
    return False

In [11]:
filtered_results = [g for g in time_results if footprint_covers(g, aoi)]

print(f"Original granules: {len(time_results)} | Valid granules: {len(filtered_results)}")

Original granules: 24 | Valid granules: 4


In [12]:
# Remove reprocessing duplicates: same acquisition, different processing ID and counter
by_acq = {}
for g in filtered_results:
    name = g['meta']['native-id']
    by_acq.setdefault('_'.join(name.split('_')[:-2]), []).append(g)

for key, items in by_acq.items():
    if len(items) > 1:
        print("duplicates:", sorted(g['meta']['native-id'][-20:] for g in items))

# Keep one per acquisition. Sorting by name picks the highest processing ID, but check the
# printout above to confirm that is the reprocessed version you want.
filtered_results = [sorted(items, key=lambda g: g['meta']['native-id'])[-1] for items in by_acq.values()]
print(f"Unique granules: {len(filtered_results)}")

Unique granules: 4


In [13]:
# Open the granules (data is read in small pieces later, not all at once)
file_objs = earthaccess.open(filtered_results)

def add_time_dimension(ds, filename):
    # The acquisition start time is in the original filename from the CMR metadata
    file_date = filename.split("_")[-4][:15]
    # Convert the date string to a datetime object
    time_value = datetime.strptime(file_date, "%Y%m%dT%H%M%S")
    # Assign the time coordinate to the dataset
    ds = ds.assign_coords(time=time_value)
    return ds.expand_dims('time')

QUEUEING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/4 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/4 [00:00<?, ?it/s]

In [14]:
datasets = []
for file_obj, granule in zip(file_objs, filtered_results):
    # chunks={} keeps the data on disk until a result is requested, avoiding memory crashes
    dataset = xr.open_dataset(file_obj, engine='h5netcdf', chunks={})
    datasets.append(add_time_dimension(dataset, granule['meta']['native-id']))

In [15]:
# sorting the time dimension chronologically
datasets.sort(key=lambda ds: ds.time.values[0])

In [16]:
# Concatenate. join='outer' builds the union of the tile grids, which stays one tile in size
#    if all granules cover the same tile.
ds2 = xr.concat(datasets, dim='time', data_vars='minimal',
                coords='minimal', compat='override', join='outer')

# Rechunk to a consistent layout
ds2 = ds2.chunk({'time': -1, 'x': 'auto', 'y': 'auto'})
print(ds2.sizes)
print(ds2.wse.notnull().sum(dim=['x', 'y']).values)   # valid pixels per time step
ds2

Frozen({'time': 4, 'y': 1549, 'x': 1549})
[227753 259410 249121 221088]


<xarray.Dataset> Size: 2GB
Dimensions:                  (time: 4, y: 1549, x: 1549)
Coordinates:
  * time                     (time) datetime64[us] 32B 2024-02-01T07:50:48 .....
  * y                        (y) float64 12kB 3.9e+06 3.901e+06 ... 4.055e+06
  * x                        (x) float64 12kB 6.788e+05 6.789e+05 ... 8.336e+05
Data variables: (12/39)
    crs                      (time) object 32B dask.array<chunksize=(4,), meta=np.ndarray>
    longitude                (time, y, x) float64 77MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    latitude                 (time, y, x) float64 77MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    wse                      (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    wse_qual                 (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    wse_qual_bitwise         (time, y, x) float64 77MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    ...                       ...
    load_tide_fes            (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    load_tide_got            (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    pole_tide                (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    model_dry_tropo_cor      (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    model_wet_tropo_cor      (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
    iono_cor_gim_ka          (time, y, x) float32 38MB dask.array<chunksize=(4, 1549, 1549), meta=np.ndarray>
Attributes: (12/49)
    Conventions:                   CF-1.7
    title:                         Level 2 KaRIn High Rate Raster Data Product
    institution:                   JPL
    source:                        Ka-band radar interferometer
    history:                       2025-11-25T23:32:40Z : Creation
    platform:                      SWOT
    ...                            ...
    utm_zone_num:                  11
    mgrs_latitude_band:            S
    x_min:                         680100.0
    x_max:                         829300.0
    y_min:                         3903300.0
    y_max:                         4052400.0

In [17]:
timeplot = ds2.wse.hvplot.image(y='y', x='x', rasterize=True)
timeplot.opts(width=700, height=500, colorbar=True)

:DynamicMap   [time]
   :Image   [x,y]   (water surface elevation above geoid)

Let's plot the `wse` quality flag, `wse_qual` which ranges 0-3 where 0=good, 1=suspect, 2=degraded, 3=bad (as described when printing variable with `xarray`).

In [18]:
timeplot = ds2.wse_qual.hvplot.image(y='y', x='x')
timeplot.opts(width=700, height=500, colorbar=True)

:DynamicMap   [time]
   :Image   [x,y]   (summary quality indicator for the water surface elevation)

### Masking a variable with its quaility flag

In [19]:
variable_to_mask = ds2['wse']
mask_variable = ds2['wse_qual']

In [20]:
# Define the condition for masking based on the range of the quaility flag
mask_condition = mask_variable <2

masked_variable = variable_to_mask.where(mask_condition)

In [21]:
# Update the masked variable in the dataset
ds2['wse'] = masked_variable

ds2['wse'].hvplot.image(y='y', x='x').opts(width=700, height=500, colorbar=True)

:DynamicMap   [time]
   :Image   [x,y]   (water surface elevation above geoid)

Our end product is a time series of the data showing only the values where the quality flag is either good (0) or suspect (1).

### Appendix: Alternate Plot

In [22]:
# # Alternate plotting with matplotlib
# %matplotlib inline

# import matplotlib.pyplot as plt
# from matplotlib import animation
# from matplotlib.animation import FuncAnimation, PillowWriter
# from IPython.display import display, Image, HTML

# variable_name = 'wse'
# data = ds2[variable_name]

# fig, ax = plt.subplots(figsize=(10, 8))
# fig.set_tight_layout({'rect': [0.01, 0.01, 1.0, 1.0]})

# contour = ax.contourf(data.isel(time=0), cmap='viridis')
# cbar = plt.colorbar(contour)
# cbar.set_label('Water Surface Elevation (meters)', fontsize=14) 
# times = ds2.time.values

# # Function to update the plot for each time step
# def update(frame):
#     ax.clear()
#     contour = ax.contourf(data.isel(time=frame), cmap='viridis')
#     formatted_time = str(times[frame])[:-7]
#     ax.set_title(f'Date: {formatted_time}')
#     ax.set_xlabel('Longitude', fontsize=14)
#     ax.set_ylabel('Latitude', fontsize=14)
#     ax.text(0.5, 1.05, 'SWOT Raster 100M Lake Mead Reservoir', transform=ax.transAxes, ha='center', fontsize=14)
#     return contour,

# # Creating a gif animation
# ani = animation.FuncAnimation(fig, update, repeat=True, frames=len(data['time']), blit=True, interval=3000)

# output = ('time_series.gif')
# ani.save(output, writer='pillow', fps=.5)

# with open(output,'rb') as f:
#     display(Image(data=f.read(), format='gif'))

# plt.close(fig)
# ds2.close()